In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import f_oneway
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import matplotlib.pyplot as plt

In [ ]:
data_mat = ['ssim_matrix_1_1_imgxavg.npy', 'euclidean_matrix_1_1_imgxavg.npy', 'hue_corr_1_1_imgxavg.npy']

for j in data_mat:
    print(j[:-23])

In [ ]:
data_mat = ['ssim_matrix_1_1_imgxavg.npy', 'euclidean_matrix_1_1_imgxavg.npy', 'hue_corr_1_1_imgxavg.npy']

for j in data_mat:

    data_name = j[:-23]
    # Load the data
    data = np.load(j)

    # Check the shape of the data
    print(data.shape)  # Should be (50, 64)

    # Prepare data for ANOVA
    # Create a DataFrame with columns for magnitude, filter, and SSIM
    magnitude = np.repeat(np.arange(data.shape[0]), data.shape[1])
    filter_id = np.tile(np.arange(data.shape[1]), data.shape[0])
    ssim_values = data.flatten()

    df = pd.DataFrame({'Magnitude': magnitude, 'Filter': filter_id, 'SSIM': ssim_values})

    # Perform ANOVA
    anova_result = f_oneway(*(df[df['Filter'] == i]['SSIM'] for i in range(64)))
    print('ANOVA result:', anova_result)

    # If ANOVA is significant, perform post-hoc tests
    if anova_result.pvalue < 0.05:
        tukey_result = pairwise_tukeyhsd(df['SSIM'], df['Filter'])
        print(tukey_result)
        # Plot the results
        tukey_result.plot_simultaneous()
        plt.show()
    else:
        print("No significant differences found among the filters.")

    # Linear regression for each filter to check if SSIM drops with increasing magnitude
    significant_filters = []
    for filter_idx in range(64):
        filter_data = df[df['Filter'] == filter_idx]
        model = ols('SSIM ~ Magnitude', data=filter_data).fit()
        print(f'Filter {filter_idx} linear regression result:')
        print(model.summary())
        # Check if the slope (coefficient of Magnitude) is significantly different from zero
        if model.pvalues['Magnitude'] < 0.05:
            significant_filters.append(filter_idx)

    print('Filters with significant SSIM drop as magnitude increases:', significant_filters)

    # For further statistical insights, we might also want to look at the means and standard deviations
    filter_means = df.groupby('Filter')['SSIM'].mean()
    filter_std = df.groupby('Filter')['SSIM'].std()
    print('Filter Means:\n', filter_means)
    print('Filter Standard Deviations:\n', filter_std)
